# RL Group Project

## Clinical Treatment Optimisation: Sepsis ICU Management

**Master in Data Science & Advanced Analytics — Reinforcement Learning Course**

This project is structured in two stages of increasing complexity.

- In **Configuration A**, you will work with a tabular Sepsis MDP, where the
  state and action spaces are small enough to apply classical RL methods
  directly.

- In **Configuration B**, you will move to a continuous-observation ICU
  environment that is clinically grounded and significantly more challenging.

Three realistic failure modes are present in Configuration B, each reflecting a
real scenario encountered in clinical AI deployments. The first is episodic
observation noise, where monitoring equipment occasionally malfunctions. The
second is episodic missing observations, representing situations where lab
results are simply unavailable for an entire episode. The third is acute
clinical events, which are sudden and irreversible patient deteriorations that
occur independently of any treatment given.

---

### Group Members

Name of the Group: Group Q

```
Student 1: Diogo Carvalho - 20221935
Student 2: Ricardo Pereira - 20250343
Student 3: Yehor Malakhov - 20221691
```


# Config B: RL on the Clinical ICU-Sepsis Environment

In Config A, patient state was represented as a discrete integer produced by
discretising a set of clinical measurements into a small number of categories.
**Config B uses the full ICU-Sepsis environment**, also built from MIMIC-III
data, but with two fundamental changes that make the problem substantially
harder.

**Change 1 — Continuous observations.** The agent now receives a
**47-dimensional continuous feature vector** instead of a single discrete index.
This vector contains the actual normalised physiological measurements used in
the original Komorowski et al. (2018) AI Clinician study, including SOFA score,
heart rate, lactate, blood pressure, creatinine, and 42 other clinical
variables.

With continuous observations, a tabular Q-table is no longer feasible: it would
require one entry per unique float vector, making it effectively infinite.

**Change 2 — Clinical reality wrappers.** Config B injects three orthogonal
failure modes that reflect challenges faced by real clinical AI deployments.

The first wrapper, `EpisodicNoisyObsEnv`, models episodic monitor malfunction.
When active, the observations received by the agent are corrupted by noise for
the entire episode, testing robustness to measurement error.

The second wrapper, `EpisodicMissingObsEnv`, models situations where lab results
are unavailable for a full episode. This tests how well the agent handles
partial observability.

The third wrapper, `AcuteEventEnv`, introduces rare, sudden patient
deterioration events such as cardiac arrest or acute organ failure. These occur
independently of any treatment decision and represent irreducible stochasticity
in the environment.

Key environment properties for Config B:

- **Actions**: 25 total (5 vasopressor levels × 5 IV fluid dose levels)
- **Reward**: +1.0 at survival, 0.0 at death, plus a small treatment intensity
  penalty (lam = 0.02)
- **Observation**: `Box(47,)`, a normalised physiological feature vector,
  potentially noisy or incomplete

## Setup and Imports


In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from stable_baselines3 import DQN, PPO

from config_b import tune_dqn, tune_ppo
from envs.env_setup import (
    GAMMA,
)
from envs.wrappers import (
    make_clinical_env,
)
from utils import EpisodeReturnsCallback, evaluate_sb3_fixed_seeds

In [17]:
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

os.makedirs("logs", exist_ok=True)

SEED = 42
np.random.seed(SEED)

In [18]:
def run_random_baseline_clinical(n_episodes=1000, seed=SEED):
    env_clinical = make_clinical_env()
    obs, info = env_clinical.reset(seed=seed)

    print(f"Observation space : {env_clinical.observation_space}")
    print(f"Action space      : {env_clinical.action_space}")
    print(f"Info keys         : {list(info.keys())}")
    print()

    clinical_rand_returns = []

    env_eval = make_clinical_env()
    for _ep in range(1000):
        obs, info = env_eval.reset(seed=np.random.randint(100_000))
        total_r, done = 0.0, False

        while not done:
            obs, r, te, tr, info = env_eval.step(env_eval.action_space.sample())
            total_r += r
            done = te or tr

        clinical_rand_returns.append(total_r)

    env_eval.close()
    env_clinical.close()
    return np.array(clinical_rand_returns)


rand_returns = run_random_baseline_clinical()

print(f"Random agent ({len(rand_returns)} episodes)")
print(f"  Overall mean return   : {np.mean(rand_returns):.4f}")
print(
    f"  Overall survival rate : {np.mean(np.array(rand_returns) > 0) * 100:.1f}%"
)

Observation space : Box(-inf, inf, (47,), float32)
Action space      : Discrete(25)
Info keys         : ['admissible_actions', 'state_vector', 'sofa_score', 'noisy_episode', 'missing_features']

Random agent (1000 episodes)
  Overall mean return   : 0.5720
  Overall survival rate : 66.3%


## DQN


In [3]:
dqn_study = tune_dqn(
    make_clinical_env,
    gamma=GAMMA,
    n_trials=30,
    eval_freq=500000,
    n_eval_episodes=1000,
    timesteps=2000000,
    seed=SEED,
    verbose=0,
)

[I 2026-06-05 02:20:57,603] A new study created in memory with name: no-name-2a9c2ef8-9052-4c1a-b992-38dc38af1390
Best trial: 0. Best value: 0.63899:   3%|▎         | 1/30 [12:13<5:54:42, 733.87s/it]

[I 2026-06-05 02:33:11,444] Trial 0 finished with value: 0.6389900020621717 and parameters: {'learning_rate': 0.0001329291894316216, 'buffer_size': 475407, 'learning_starts': 7588, 'batch_size': 160, 'train_freq': 16, 'tau': 0.06358358856676251, 'target_update_interval': 709, 'exploration_fraction': 0.020086402204943198, 'exploration_final_eps': 0.0972918866945795}. Best is trial 0 with value: 0.6389900020621717.


Best trial: 0. Best value: 0.63899:   7%|▋         | 2/30 [31:41<7:41:32, 989.02s/it]

[I 2026-06-05 02:52:39,054] Trial 1 finished with value: 0.5748775016451254 and parameters: {'learning_rate': 0.00314288089084011, 'buffer_size': 106957, 'learning_starts': 2636, 'batch_size': 64, 'train_freq': 4, 'tau': 0.06847920095574778, 'target_update_interval': 140, 'exploration_fraction': 0.1531508777822569, 'exploration_final_eps': 0.04297256589643226}. Best is trial 0 with value: 0.6389900020621717.


Best trial: 2. Best value: 0.646018:  10%|█         | 3/30 [45:22<6:50:30, 912.26s/it]

...

[I 2026-06-05 14:58:06,267] Trial 27 finished with value: 0.4716375001128763 and parameters: {'learning_rate': 0.007321428778380786, 'buffer_size': 369211, 'learning_starts': 5989, 'batch_size': 160, 'train_freq': 16, 'tau': 0.001104537501931838, 'target_update_interval': 117, 'exploration_fraction': 0.03254129459065885, 'exploration_final_eps': 0.013665592208707313}. Best is trial 24 with value: 0.6790975024444051.


Best trial: 24. Best value: 0.679098:  97%|█████████▋| 29/30 [13:24:02<34:21, 2061.81s/it]

[I 2026-06-05 15:45:00,466] Trial 28 finished with value: 0.5814425016706809 and parameters: {'learning_rate': 0.0036845270736218664, 'buffer_size': 352125, 'learning_starts': 5268, 'batch_size': 32, 'train_freq': 1, 'tau': 0.015686071966628894, 'target_update_interval': 616, 'exploration_fraction': 0.32119588892514545, 'exploration_final_eps': 0.014077360879484007}. Best is trial 24 with value: 0.6790975024444051.


Best trial: 24. Best value: 0.679098: 100%|██████████| 30/30 [14:11:02<00:00, 1702.09s/it]

[I 2026-06-05 16:32:00,341] Trial 29 finished with value: 0.6327900022733957 and parameters: {'learning_rate': 0.0001329957747943182, 'buffer_size': 313304, 'learning_starts': 5528, 'batch_size': 224, 'train_freq': 1, 'tau': 0.001200974902063027, 'target_update_interval': 586, 'exploration_fraction': 0.4707128182982292, 'exploration_final_eps': 0.061792676008829116}. Best is trial 24 with value: 0.6790975024444051.


In [4]:
dqn_study.best_trial.params

{'learning_rate': 7.238086291181395e-05,
 'buffer_size': 295826,
 'learning_starts': 1274,
 'batch_size': 32,
 'train_freq': 1,
 'tau': 0.20416470207182763,
 'target_update_interval': 216,
 'exploration_fraction': 0.31521633315131015,
 'exploration_final_eps': 0.01768127184943912}

In [23]:
pl.DataFrame(dqn_study.best_trial.params).write_csv(
    "logs/dqn_best_trial_params.csv"
)

In [6]:
dqn_episode_returns_callback = EpisodeReturnsCallback()

In [8]:
dqn_best = DQN(
    "MlpPolicy",
    make_clinical_env(),
    learning_rate=dqn_study.best_trial.params["learning_rate"],
    buffer_size=dqn_study.best_trial.params["buffer_size"],
    learning_starts=dqn_study.best_trial.params["learning_starts"],
    batch_size=dqn_study.best_trial.params["batch_size"],
    train_freq=dqn_study.best_trial.params["train_freq"],
    tau=dqn_study.best_trial.params["tau"],
    gamma=GAMMA,
    target_update_interval=dqn_study.best_trial.params[
        "target_update_interval"
    ],
    exploration_fraction=dqn_study.best_trial.params["exploration_fraction"],
    exploration_initial_eps=1.0,
    exploration_final_eps=dqn_study.best_trial.params["exploration_final_eps"],
    seed=SEED,
)
dqn_best.learn(total_timesteps=2000000, callback=dqn_episode_returns_callback)

In [10]:
dqn_mean_reward = evaluate_sb3_fixed_seeds(
    dqn_best,
    make_clinical_env,
    n_episodes=15000,
    first_seed=10000,
)

In [22]:
pl.DataFrame(dqn_episode_returns_callback.episode_returns).write_csv(
    "logs/dqn_episode_returns_train.csv"
)
pl.DataFrame(dqn_mean_reward).write_csv("logs/dqn_mean_return_eval.csv")

In [ ]:
print(
    f"DQN mean reward over 2M episodes: {np.array(dqn_mean_reward).mean():.4f}"
)
print(f"Survival rate: {(np.array(dqn_mean_reward) > 0).mean() * 100:.1f}%")

DQN mean reward over 2M episodes: 0.6565
Survival rate: 70.1%


## PPO


In [ ]:
ppo_study = tune_ppo(
    make_clinical_env,
    gamma=GAMMA,
    n_trials=30,
    eval_freq=500000,
    n_eval_episodes=1000,
    timesteps=2000000,
    seed=SEED,
    verbose=0,
)

[I 2026-06-05 04:01:43,805] A new study created in memory with name: no-name-8671992f-10dd-4d74-98b3-62d1283b5bb0
  0%|          | 0/30 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.612948:   3%|▎         | 1/30 [55:50<26:59:33, 3350.80s/it]

[I 2026-06-05 04:57:34,571] Trial 0 finished with value: 0.6129475023546256 and parameters: {'learning_rate': 0.0001329291894316216, 'n_steps': 128, 'batch_size': 8, 'n_epochs': 10, 'gae_lambda': 0.9665419697120591, 'clip_range': 0.14246782213565523, 'ent_coef': 0.0003511356313970409, 'vf_coef': 0.2650640588680905, 'max_grad_norm': 0.6597435289084645}. Best is trial 0 with value: 0.6129475023546256.


Best trial: 1. Best value: 0.634295:   7%|▋         | 2/30 [1:21:22<17:44:15, 2280.54s/it]

[I 2026-06-05 05:23:05,935] Trial 1 finished with value: 0.63429500289727 and parameters: {'learning_rate': 0.00037520558551242813, 'n_steps': 512, 'batch_size': 16, 'n_epochs': 7, 'gae_lambda': 0.8565030577807997, 'clip_range': 0.22150897038028766, 'ent_coef': 0.00032476735706274504, 'vf_coef': 0.1585464336867516, 'max_grad_norm': 3.5039617138009986}. Best is trial 1 with value: 0.63429500289727.


...

[I 2026-06-05 16:17:28,635] Trial 27 finished with value: 0.6507000046288595 and parameters: {'learning_rate': 2.2722376351800995e-05, 'n_steps': 2048, 'batch_size': 64, 'n_epochs': 3, 'gae_lambda': 0.9857830357390694, 'clip_range': 0.10863198239011523, 'ent_coef': 0.047144316216931145, 'vf_coef': 0.5749309981776699, 'max_grad_norm': 3.9277679131728775}. Best is trial 19 with value: 0.6726775041832589.


Best trial: 19. Best value: 0.672678:  97%|█████████▋| 29/30 [12:31:00<20:49, 1249.12s/it]

[I 2026-06-05 16:32:43,813] Trial 28 finished with value: 0.6511575050307438 and parameters: {'learning_rate': 1.664905907163967e-05, 'n_steps': 256, 'batch_size': 32, 'n_epochs': 5, 'gae_lambda': 0.9830576077707183, 'clip_range': 0.27805275677818325, 'ent_coef': 0.0023279318292339347, 'vf_coef': 0.658119338021383, 'max_grad_norm': 0.6154008507029678}. Best is trial 19 with value: 0.6726775041832589.


Best trial: 19. Best value: 0.672678: 100%|██████████| 30/30 [13:53:20<00:00, 1666.70s/it]

[I 2026-06-05 17:55:04,693] Trial 29 finished with value: 0.645602502942551 and parameters: {'learning_rate': 3.667443899176896e-05, 'n_steps': 2048, 'batch_size': 4, 'n_epochs': 8, 'gae_lambda': 0.8727663715082847, 'clip_range': 0.2821854368987685, 'ent_coef': 0.029350244859083902, 'vf_coef': 0.9548199219627316, 'max_grad_norm': 1.9656728609356706}. Best is trial 19 with value: 0.6726775041832589.


In [ ]:
ppo_study.best_trial.params

{'learning_rate': 6.91515136601163e-05,
 'n_steps': 128,
 'batch_size': 16,
 'n_epochs': 3,
 'gae_lambda': 0.8557020323246558,
 'clip_range': 0.27109211680220147,
 'ent_coef': 0.012911407200549929,
 'vf_coef': 0.5267564461785927,
 'max_grad_norm': 0.386525981643631}

In [ ]:
pl.DataFrame(ppo_study.best_trial.params).write_csv(
    "logs/ppo_best_trial_params.csv"
)

In [ ]:
episode_returns_callback = EpisodeReturnsCallback()

In [ ]:
ppo_thing = PPO(
    policy="MlpPolicy",
    env=make_clinical_env(),
    learning_rate=ppo_study.best_trial.params["learning_rate"],
    n_steps=ppo_study.best_trial.params["n_steps"],
    batch_size=ppo_study.best_trial.params["batch_size"],
    n_epochs=ppo_study.best_trial.params["n_epochs"],
    gamma=GAMMA,
    gae_lambda=ppo_study.best_trial.params["gae_lambda"],
    clip_range=ppo_study.best_trial.params["clip_range"],
    ent_coef=ppo_study.best_trial.params["ent_coef"],
    vf_coef=ppo_study.best_trial.params["vf_coef"],
    max_grad_norm=ppo_study.best_trial.params["max_grad_norm"],
    seed=SEED,
)

ppo_thing.learn(2_000_000, callback=episode_returns_callback)

In [ ]:
ppo_mean_reward = evaluate_sb3_fixed_seeds(
    ppo_thing,
    make_clinical_env,
    n_episodes=15000,
)

In [ ]:
pl.DataFrame(episode_returns_callback.episode_returns).write_csv(
    "logs/ppo_episode_returns_train.csv"
)
pl.DataFrame(ppo_mean_reward).write_csv("logs/ppo_mean_return_eval.csv")

In [ ]:
print(
    f"PPO mean reward over 2M episodes: {np.array(ppo_mean_reward).mean():.4f}"
)
print(f"Survival rate: {(np.array(ppo_mean_reward) > 0).mean() * 100:.1f}%")

PPO mean reward over 2M episodes: 0.6688
Survival rate: 68.5%
